# Word-Level Classification Baseline (from hand keypoints)

**Task:** predict which of the ~86 words was signed, using the word itself (video filename) as the label. This needs no letter-level annotation -- the label already exists in your CSV.

**Result (spoiler, see bottom):** both a signer-held-out split and a signer-shared split land near chance (~1-4%). This isn't a bug -- with only ~3-4 examples per word class, no classifier can learn 86-way word identity from this little data per class. The real takeaway is that **word-level classification is a poor task given how this dataset is structured** (each word signed once per signer); **letter-level classification is much better posed**, since individual letters repeat across many words. This notebook documents that finding as a stepping stone, not a final result.


In [1]:
import pandas as pd
import numpy as np
from sklearn.ensemble import RandomForestClassifier
from sklearn.preprocessing import LabelEncoder
from sklearn.model_selection import train_test_split
from sklearn.metrics import accuracy_score, top_k_accuracy_score
from collections import Counter

CSV_PATH = "hand_keypoints_features.csv"  # edit for your machine
df = pd.read_csv(CSV_PATH)
DIST_COLS = [c for c in df.columns if c.startswith('d_')]
KP_COLS_X = [f"kp{i}_x" for i in range(21)]
KP_COLS_Y = [f"kp{i}_y" for i in range(21)]
print(df.shape)


(72087, 76)


## Feature engineering

Each video has a different number of frames, so we aggregate into one fixed-length vector per video: mean/std of each geometric distance feature, motion-intensity summary stats, detection quality, and video length.

In [2]:
def video_motion_intensity(sub: pd.DataFrame) -> np.ndarray:
    xs = sub[KP_COLS_X].to_numpy()
    ys = sub[KP_COLS_Y].to_numpy()
    dx = np.diff(xs, axis=0)
    dy = np.diff(ys, axis=0)
    disp = np.sqrt(dx**2 + dy**2).mean(axis=1)
    return np.concatenate([[0.0], disp])


def build_feature_vector(sub: pd.DataFrame) -> np.ndarray:
    feats = []
    for c in DIST_COLS:
        vals = sub[c].to_numpy()
        feats.append(np.nanmean(vals))
        feats.append(np.nanstd(vals))
    motion = video_motion_intensity(sub)
    feats += [motion.mean(), motion.std(), motion.max()]
    feats.append(sub['hand_detected'].mean())
    feats.append(sub['mean_confidence'].mean())
    feats.append(len(sub))
    return np.array(feats, dtype=float)


def build_dataset(df: pd.DataFrame, min_detection_rate: float = 0.5):
    rows = []
    for (signer, video), sub in df.groupby(['signer_id', 'video_id']):
        if sub['hand_detected'].mean() < min_detection_rate:
            continue
        sub = sub.sort_values('frame_id')
        feat = build_feature_vector(sub)
        rows.append((signer, video, feat))
    return rows


rows = build_dataset(df)
print(f"Built {len(rows)} (signer, video) feature vectors, dim={len(rows[0][2])}")

le = LabelEncoder()
le.fit([r[1] for r in rows])
signers = sorted(set(r[0] for r in rows))


Built 317 (signer, video) feature vectors, dim=56


## Leave-One-Signer-Out (LOSO) classification

Train a RandomForest on 4 signers, test on the 5th (only evaluating on word classes actually seen during training -- a classifier can't predict a class it's never seen).

In [3]:
results = {}
for held_out in signers:
    train_rows = [r for r in rows if r[0] != held_out]
    test_rows = [r for r in rows if r[0] == held_out]

    X_train = np.stack([r[2] for r in train_rows])
    y_train_labels = [r[1] for r in train_rows]
    X_test = np.stack([r[2] for r in test_rows])
    y_test_labels = [r[1] for r in test_rows]

    train_classes = set(y_train_labels)
    keep = [i for i, w in enumerate(y_test_labels) if w in train_classes]
    dropped = len(y_test_labels) - len(keep)

    X_test_f = X_test[keep]
    y_test_f = [y_test_labels[i] for i in keep]

    y_train = le.transform(y_train_labels)
    y_test = le.transform(y_test_f)

    clf = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
    clf.fit(X_train, y_train)
    y_pred = clf.predict(X_test_f)
    acc = accuracy_score(y_test, y_pred)

    probs = clf.predict_proba(X_test_f)
    clf_classes = clf.classes_
    top5 = top_k_accuracy_score(y_test, probs, k=5, labels=clf_classes) if len(clf_classes) > 5 else float('nan')

    results[held_out] = acc
    print(f"held out {held_out:<12} test_n={len(y_test_f):<4} "
          f"(dropped {dropped} unseen-class videos) "
          f"top1_acc={acc:.3f}  top5_acc={top5:.3f}")

n_classes = len(le.classes_)
print(f"\nMean LOSO top-1 accuracy: {np.mean(list(results.values())):.3f}")
print(f"(chance level with {n_classes} classes ~= {1/n_classes:.3f})")


held out signer1-F    test_n=66   (dropped 4 unseen-class videos) top1_acc=0.015  top5_acc=0.076
held out signer2-F    test_n=69   (dropped 1 unseen-class videos) top1_acc=0.029  top5_acc=0.174
held out signer3-F    test_n=65   (dropped 2 unseen-class videos) top1_acc=0.062  top5_acc=0.169
held out signer4-L    test_n=67   (dropped 3 unseen-class videos) top1_acc=0.015  top5_acc=0.119
held out signer5-R    test_n=35   (dropped 5 unseen-class videos) top1_acc=0.057  top5_acc=0.114

Mean LOSO top-1 accuracy: 0.036
(chance level with 86 classes ~= 0.012)


## Stratified (signer-shared) split, for comparison

In [4]:
word_counts = Counter(r[1] for r in rows)
rows_ok = [r for r in rows if word_counts[r[1]] >= 2]
print(f"(dropped {len(rows) - len(rows_ok)} videos whose word has <2 total good samples, needed for stratified splitting)")

X_all = np.stack([r[2] for r in rows_ok])
y_all_labels = [r[1] for r in rows_ok]
y_all = le.transform(y_all_labels)

X_tr, X_te, y_tr, y_te = train_test_split(X_all, y_all, test_size=0.25, random_state=42, stratify=y_all)
clf2 = RandomForestClassifier(n_estimators=300, random_state=42, n_jobs=-1)
clf2.fit(X_tr, y_tr)
acc2 = accuracy_score(y_te, clf2.predict(X_te))
probs2 = clf2.predict_proba(X_te)
top5_2 = top_k_accuracy_score(y_te, probs2, k=5, labels=clf2.classes_)
print(f"stratified split: test_n={len(y_te)} top1_acc={acc2:.3f} top5_acc={top5_2:.3f}")


(dropped 15 videos whose word has <2 total good samples, needed for stratified splitting)
stratified split: test_n=76 top1_acc=0.013 top5_acc=0.039


c:\Users\H.P\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:98: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_true = type_of_target(y_true, input_name="y_true")
c:\Users\H.P\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_classification.py:99: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  type_pred = type_of_target(y_pred, input_name="y_pred")
c:\Users\H.P\AppData\Local\Programs\Python\Python312\Lib\site-packages\sklearn\metrics\_ranking.py:1998: UserWarning: The number of unique classes is greater than 50% of the number of samples. `y` could represent a regression problem, not a classification problem.
  y_type = type_of_target(y_true, input_name="y_true")


## Interpretation

Both splits land near chance. This is **not** primarily a signer-generalization story (the LOSO vs. stratified gap the Illusion-to-Reality paper reports) -- it's a **data-scarcity** story: with ~86 classes and only ~3-4 samples each, no classifier can learn this reliably regardless of split strategy.

**Implication for what to build next:** word-level classification is structurally low-data (each word signed once per signer). Letter-level classification is much better posed -- individual letters repeat dozens to hundreds of times across the corpus (e.g. 'ی' appears 455 times per the PSL-CFRT paper's frequency table). Getting frame-level letter labels (from Kamran's annotated set, or by building our own from segment boundaries + the noun-to-letter-sequence mapping) should be the next priority over further tuning this word classifier.
